# HW2.2: Storing OLTP Events in a Data Lake

Creates a personal S3 bucket (`dsan6000-<NetID>`) and uploads the locally-downloaded hourly `.parquet` files into a `wikipedia-hourly/` "subfolder" of that bucket.

In [ ]:
import glob
import os

import boto3
from botocore.exceptions import ClientError


In [ ]:
NETID = "jh2732"
BUCKET_NAME = f"dsan6000-{NETID}"
LOCAL_DATA_DIR = "data"
DEST_PREFIX = "wikipedia-hourly"


In [ ]:
s3_client = boto3.client("s3")
region = s3_client.meta.region_name

try:
    if region == "us-east-1":
        s3_client.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": region},
        )
    print(f"Created bucket: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "BucketAlreadyOwnedByYou":
        print(f"Bucket already exists: {BUCKET_NAME}")
    else:
        raise


In [ ]:
local_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "*.parquet")))
print(f"Found {len(local_files)} local parquet files to upload")
local_files


In [ ]:
for local_path in local_files:
    filename = os.path.basename(local_path)
    s3_key = f"{DEST_PREFIX}/{filename}"
    s3_client.upload_file(local_path, BUCKET_NAME, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET_NAME}/{s3_key}")


In [ ]:
response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{DEST_PREFIX}/")
uploaded_keys = [obj["Key"] for obj in response.get("Contents", [])]
print(f"{len(uploaded_keys)} objects now in s3://{BUCKET_NAME}/{DEST_PREFIX}/")
uploaded_keys
